# U-Net Segmentation Training - Google Colab Orchestrator

**Orchestrator only.** All training logic lives in the main repository files
(`train.py`, `utils/data_loading.py`, `unet/`, etc.). This notebook simply clones
the repo, installs dependencies, and calls into the existing code.

---

| Item | Value |
|------|-------|
| Source repo | `https://github.com/HaikalFK/segmentasi-unet.git` |
| Dataset | Plant Phenotyping (Kaggle) |
| Classes | 22 (auto-detected) |
| Runtime | **GPU** (T4, V100, or A100) |

---
## 1. Install Dependencies

PyTorch with CUDA is pre-installed in Colab. We only need to install
the project-specific dependencies from `requirements.txt`.

In [ ]:
# Verify GPU is available
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device: {torch.cuda.get_device_name(0)}')

In [ ]:
# Clone the repository (or pull latest if already cloned)
import os
from pathlib import Path

REPO_URL = 'https://github.com/HaikalFK/segmentasi-unet.git'
REPO_DIR = Path('/content/segmentasi-unet')

if not REPO_DIR.exists():
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repository already cloned. Fetching latest...')
    %cd {REPO_DIR}
    !git fetch --all
    !git reset --hard origin/main

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install project dependencies (version ranges to avoid build failures on Colab)
!pip install --quiet --upgrade pip
!pip install --quiet 'matplotlib>=3.6.0' 'numpy>=1.24.0' 'Pillow>=9.3.0' 'tqdm>=4.64.0' 'wandb>=0.18.0' kagglehub

# Verify imports
from utils.data_loading import BasicDataset
from unet import UNet
print('All imports OK')

---
## 2. Download Dataset

Downloads from Kaggle and organizes into `data/imgs/` and `data/masks/`.
Uses the existing `data/download_dataset.py` script.

In [ ]:
!python data/download_dataset.py

---
## 3. Run Training

All parameters are passed as CLI arguments to `train.py`.
Edit the variables below to configure training.

In [ ]:
# ============================================================
# TRAINING CONFIGURATION
# Edit these values as needed.
# ============================================================
EPOCHS        = 100
BATCH_SIZE    = 8
SCALE         = 0.5
LEARNING_RATE = 1e-5
VALIDATION    = 10.0
AMP           = True
BILINEAR      = False

print('Configuration:')
print(f'  Epochs:    {EPOCHS}')
print(f'  Batch:     {BATCH_SIZE}')
print(f'  Scale:     {SCALE}')
print(f'  AMP:       {AMP}')
print(f'  Bilinear:  {BILINEAR}')

In [ ]:
# Build and execute the training command
cmd = (
    f'python train.py'
    f' --classes 21'
    f' --epochs {EPOCHS}'
    f' --batch-size {BATCH_SIZE}'
    f' --scale {SCALE}'
    f' --learning-rate {LEARNING_RATE}'
    f' --validation {VALIDATION}'
)

if AMP:
    cmd += ' --amp'
if BILINEAR:
    cmd += ' --bilinear'

print(f'Command: {cmd}')
print('=' * 70)
!{cmd}

---
## 4. Download Checkpoints

After training completes, download the checkpoint files to your local machine.

In [ ]:
from pathlib import Path
checkpoints = sorted(Path('checkpoints').glob('*.pth'))
print(f'Checkpoints found: {len(checkpoints)}')
for ckpt in checkpoints:
    size_mb = ckpt.stat().st_size / (1024 * 1024)
    print(f'  {ckpt.name}  ({size_mb:.2f} MB)')

In [ ]:
# Download the last checkpoint to local machine
from google.colab import files

if checkpoints:
    latest = checkpoints[-1]
    files.download(str(latest))
else:
    print('No checkpoints found. Run training first.')

---
## 5. Evaluate Model

Run evaluation using `evaluate.py` against a trained checkpoint.

In [ ]:
CHECKPOINT_PATH = 'checkpoints/checkpoint_epoch100.pth'  # adjust
!python evaluate.py --load {CHECKPOINT_PATH} --classes 21 --scale {SCALE}

---
## 6. Predict on a Sample Image

Uses `predict.py` from the main repo.

In [ ]:
CHECKPOINT_PATH = 'checkpoints/checkpoint_epoch100.pth'  # adjust
SAMPLE_IMAGE = 'data/imgs/ara2012_plant001.png'

!python predict.py --model {CHECKPOINT_PATH} --input {SAMPLE_IMAGE} --classes 21 --scale {SCALE} --viz